### VAE training for real data

V1: 2026/05/31     
V3: 2026/06/02 added balanced cluster modeling     
V4: 2026/06/04 updated preprocess function

In [ ]:
#!pip install scikit-optimize

In [1]:
import json
import os
from pathlib import Path
from math import sqrt
import xarray as xr
import pandas as pd
import numpy as np
from collections import defaultdict
from umap import UMAP
import pickle


import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt 

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn.functional as F

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MeanShift, estimate_bandwidth
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
from scipy.signal import savgol_filter, find_peaks
from scipy.stats import mode
import hdbscan
import seaborn as sns

import cv2

import gc
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args


#### Prepare training data

In [2]:


# ── just point to your folder ─────────────────────────────────────────────────
json_folder = Path(r"D:\_Dev\2026\vae_jackie\data")   # Path() + raw string (r"...")
json_files  = list(json_folder.glob("*.json"))
print(f"Found {len(json_files)} json files")

all_arrays = []
body_parts = None

for json_file in json_files:
    with open(json_file) as f:
        data = json.load(f)

    annotations = data["annotations"]
    frame_ids   = list(annotations.keys())
    n_frames    = len(frame_ids)
    first_key   = frame_ids[0]

    if body_parts is None:
        body_parts = list(annotations[first_key].keys())
    else:
        assert list(annotations[first_key].keys()) == body_parts, \
            f"Body parts mismatch in {json_file}"

    n_bodyparts = len(body_parts)
    arr = np.zeros((n_frames, n_bodyparts, 2), dtype=np.float32)

    for fi, fid in enumerate(frame_ids):
        for bi, bp in enumerate(body_parts):
            arr[fi, bi, :] = annotations[fid][bp]

    all_arrays.append(arr)
    print(f"Loaded {json_file.name} — {n_frames} frames")

# ── concatenate everything into one array ─────────────────────────────────────
arr_all = np.concatenate(all_arrays, axis=0)

da = xr.DataArray(
    arr_all,
    dims=["frame", "bodypart", "coord"],
    coords={
        "frame":    np.arange(len(arr_all)),
        "bodypart": body_parts,
        "coord":    ["x", "y"],
    },
    name="keypoints",
)

print("\nDataArray shape:", da.shape)

nc_file = "./data/exp_data.nc"
da.to_netcdf(nc_file)


Found 12 json files
Loaded 2026-05-11 10-45-46_bottom_left.json — 312589 frames
Loaded 2026-05-11 10-45-46_bottom_right.json — 314117 frames
Loaded 2026-05-11 10-45-46_top_left.json — 313777 frames
Loaded 2026-05-11 10-45-46_top_right.json — 314264 frames
Loaded 2026-05-11 10-46-09_bottom_right.json — 317041 frames
Loaded 2026-05-11 10-46-09_top_left.json — 316845 frames
Loaded 2026-05-12 11-26-37_bottom_left.json — 307915 frames
Loaded 2026-05-12 11-26-37_bottom_right.json — 308734 frames
Loaded 2026-05-12 11-26-37_top_left.json — 298745 frames
Loaded 2026-05-12 11-26-37_top_right.json — 308150 frames
Loaded 2026-05-12 11-27-00_bottom_right.json — 322637 frames
Loaded 2026-05-12 11-27-00_top_left.json — 322488 frames

DataArray shape: (3757302, 7, 2)


Load prepared dataset from local disk

In [ ]:
nc_file = "./data/exp_data.nc"
da = xr.open_dataarray(nc_file)
print("\nDataArray shape:", da.shape)

In [3]:
# V4
def preprocess(raw_sequence):
    # compute speed from raw arena movement BEFORE centering
    center_raw  = raw_sequence[:, 1, :]                        # (T, 2)
    center_diff = np.diff(center_raw, axis=0)                  # (T-1, 2)
    center_diff = np.concatenate([center_diff[:1], center_diff], axis=0)  # (T, 2)
    speed       = np.linalg.norm(center_diff, axis=-1)         # (T,)
    print(f"Speed stats: mean={speed.mean():.3f}, std={speed.std():.3f}, max={speed.max():.3f}")
    speed       = np.tile(speed[:, None, None], (1, 7, 1))     # (T, 7, 1)

    # center and heading-align
    centroid = raw_sequence[:, 1:2, :]
    centered = raw_sequence - centroid

    head  = centered[:, 0, :]
    angle = np.arctan2(head[:, 1], head[:, 0])
    cos_a = np.cos(-angle)
    sin_a = np.sin(-angle)

    x = centered[:, :, 0]
    y = centered[:, :, 1]
    x_rot = x * cos_a[:, None] - y * sin_a[:, None]
    y_rot = x * sin_a[:, None] + y * cos_a[:, None]
    aligned = np.stack([x_rot, y_rot], axis=-1)                # (T, 7, 2)

    # velocity of aligned joints (captures gait/paw swing)
    vel = np.diff(aligned, axis=0)
    vel = np.concatenate([vel[:1], vel], axis=0)               # (T, 7, 2)

    # angular velocity (captures turning)
    ang_vel = np.diff(angle, axis=0)
    ang_vel = np.concatenate([ang_vel[:1], ang_vel])
    ang_vel = np.tile(ang_vel[:, None, None], (1, 7, 1))       # (T, 7, 1)

    return np.concatenate([aligned, vel, ang_vel, speed], axis=-1)  # (T, 7, 6)

raw_processed = preprocess(da.values)

Speed stats: mean=1.558, std=3.114, max=508.398


In [4]:
n_frames, n_joints, n_coords = raw_processed.shape
da_scaled = np.zeros_like(raw_processed)
scalers = []

for j in range(n_joints):
    joint_scalers = []
    for c in range(n_coords):
        scaler = StandardScaler()
        da_scaled[:, j, c] = scaler.fit_transform(
            raw_processed[:, j, c].reshape(-1, 1)
        ).squeeze()
        joint_scalers.append(scaler)
    scalers.append(joint_scalers)

raw_processed = da_scaled

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

#### Classes and functions for modeling

In [6]:
# ============================================================
# MODEL
# ============================================================

class HierarchicalRAE(nn.Module):
    def __init__(self,
                 joint_dim,
                 joint_embed=32,
                 pose_embed=128,
                 hidden_dim=256,
                 latent_dim=64,
                 num_joints=7):
        super().__init__()
        self.num_joints = num_joints

        # --- Encoder ---
        self.joint_encoder = nn.Sequential(
            nn.Linear(joint_dim, joint_embed),
            nn.Tanh(),
            nn.Linear(joint_embed, joint_embed)
        )
        self.pose_encoder = nn.Sequential(
            nn.Linear(num_joints * joint_embed, pose_embed),
            nn.Tanh()
        )
        self.encoder_rnn = nn.LSTM(pose_embed, hidden_dim, batch_first=True)
        self.fc_latent    = nn.Linear(hidden_dim, latent_dim)

        # --- Decoder ---
        self.fc_decode_h  = nn.Linear(latent_dim, hidden_dim)
        self.fc_decode_c  = nn.Linear(latent_dim, hidden_dim)
        self.decoder_rnn  = nn.LSTM(latent_dim, hidden_dim, batch_first=True)
        self.decoder_proj = nn.Linear(hidden_dim, pose_embed)
        self.pose_decoder = nn.Sequential(
            nn.Linear(pose_embed, num_joints * joint_embed),
            nn.Tanh()
        )
        self.joint_decoder = nn.Linear(joint_embed, joint_dim)

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc_decode_h.weight, gain=2.0)
        nn.init.xavier_uniform_(self.fc_decode_c.weight, gain=2.0)
        nn.init.constant_(self.fc_decode_h.bias, 0.0)
        nn.init.constant_(self.fc_decode_c.bias, 0.0)
        nn.init.xavier_uniform_(self.fc_latent.weight, gain=2.0)
        nn.init.constant_(self.fc_latent.bias, 0.0)

        for name, param in self.decoder_rnn.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param, gain=2.0)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param, gain=2.0)
            elif 'bias' in name:
                nn.init.zeros_(param)

        for name, param in self.encoder_rnn.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)

        for module in [self.joint_encoder, self.pose_encoder,
                       self.pose_decoder, self.joint_decoder,
                       self.decoder_proj]:
            if isinstance(module, nn.Sequential):
                for layer in module:
                    if isinstance(layer, nn.Linear):
                        nn.init.xavier_uniform_(layer.weight)
                        nn.init.zeros_(layer.bias)
            elif isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def encode(self, x, lengths=None):
        from torch.nn.utils.rnn import pack_padded_sequence
        B, T, J, C = x.shape

        x_flat = x.view(B * T * J, C)
        x_flat = self.joint_encoder(x_flat)
        x_enc  = x_flat.view(B, T, J, -1)
        x_enc  = x_enc.view(B, T, -1)
        x_enc  = self.pose_encoder(x_enc)

        if lengths is not None:
            packed = pack_padded_sequence(
                x_enc, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            _, (h, _) = self.encoder_rnn(packed)
        else:
            _, (h, _) = self.encoder_rnn(x_enc)

        z = self.fc_latent(h[-1])
        return z

    def decode(self, z, T):
        B   = z.shape[0]
        h   = self.fc_decode_h(z).unsqueeze(0)
        c   = self.fc_decode_c(z).unsqueeze(0)
        inp = z.unsqueeze(1).repeat(1, T, 1)
        dec, _ = self.decoder_rnn(inp, (h, c))
        dec    = self.decoder_proj(dec)
        return dec

    def forward(self, x, lengths=None):
        B, T, J, C = x.shape
        z   = self.encode(x, lengths)
        dec = self.decode(z, T)
        dec = self.pose_decoder(dec)
        dec = dec.view(B, T, J, -1)
        dec = dec.view(B * T * J, -1)
        dec = self.joint_decoder(dec)
        dec = dec.view(B, T, J, C)
        return dec, z


# ============================================================
# COLLATE FUNCTION
# ============================================================

def collate_variable_length(batch):
    lengths = [x.shape[0] for x in batch]
    B       = len(batch)
    T_max   = max(lengths)
    J, C    = batch[0].shape[1], batch[0].shape[2]

    padded = torch.zeros(B, T_max, J, C)
    for i, x in enumerate(batch):
        padded[i, :lengths[i]] = x

    return padded, torch.tensor(lengths, dtype=torch.long)


# ============================================================
# EXTRACT WINDOWS (fixed-size, Stage 1 only)
# ============================================================

def extract_nonoverlapping_windows(raw_sequence, window_size):
    n_frames = len(raw_sequence)
    windows  = []
    for start in range(0, n_frames - window_size, window_size):
        windows.append(raw_sequence[start:start + window_size])
    windows = np.array(windows)
    print(f"Extracted {len(windows)} non-overlapping windows")
    print(f"Coverage: {len(windows) * window_size}/{n_frames} frames "
          f"({100 * len(windows) * window_size / n_frames:.1f}%)")
    return windows



#### Training starts here

In [7]:
MAX_ITER = 15
EPOCHS = 2000
VW_EPOCHS = 1000
BATCH_SIZE = 256     #640   # 1280    # 1536  # 128X12
LR = 1e-3
WINDOW_SIZE = 30

PERCENTILE_RANGE = (50,75)   #(45.0, 70.0)
QUANTILE_RANGE = (0.05,0.20)   # (0.05, 0.2)
LR_RANGE = (1e-4, 1e-2)

Function definition

In [8]:

# ============================================================
# STAGE 1: Train on fixed-size windows
# ============================================================

def train_on_fixed_windows(raw_sequence, window_size=WINDOW_SIZE,
                            epochs=EPOCHS, batch_size=BATCH_SIZE,
                            lr=LR, device=device,
                            patience=30):
    windows  = extract_nonoverlapping_windows(raw_sequence, window_size)
    X_tensor = torch.tensor(windows, dtype=torch.float32)

    n_joints  = raw_sequence.shape[1]
    joint_dim = raw_sequence.shape[2]

    from torch.utils.data import TensorDataset
    dataset = TensorDataset(X_tensor)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model     = HierarchicalRAE(latent_dim=30, joint_dim=joint_dim,
                                num_joints=n_joints).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='min', factor=0.5, patience=10, verbose=False)
    loss_fn   = nn.MSELoss()

    best_loss    = float('inf')
    patience_ctr = 0
    best_weights = None
    lossgraph    = []

    print("Training on fixed windows...")
    for epoch in range(epochs):
        total_loss = 0
        for (batch,) in loader:
            batch    = batch.to(device)
            recon, _ = model(batch)
            loss     = loss_fn(recon, batch)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        avg = total_loss / len(loader)
        lossgraph.append(avg)
        scheduler.step(avg)

        if avg < best_loss - 1e-4:
            best_loss    = avg
            patience_ctr = 0
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1

        if patience_ctr >= patience:
            print(f"Early stopping at epoch {epoch+1} — best loss: {best_loss:.6f}")
            model.load_state_dict(best_weights)
            break

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} — Loss: {avg:.6f}  "
                  f"patience: {patience_ctr}/{patience}")

    return model, lossgraph


# ============================================================
# STAGE 4b: Retrain on variable-length windows
# ============================================================

def train_on_variable_windows(raw_sequence, windows,
                               epochs=VW_EPOCHS, batch_size=BATCH_SIZE,
                               lr=LR, device=device,
                               patience=30):
    
    
    
    n_joints  = raw_sequence.shape[1]
    joint_dim = raw_sequence.shape[2]

    tensor_list = [torch.tensor(w, dtype=torch.float32) for w in windows]
    loader = DataLoader(tensor_list, batch_size=batch_size, shuffle=True,
                        collate_fn=collate_variable_length)

    model     = HierarchicalRAE(latent_dim=30, joint_dim=joint_dim,
                                num_joints=n_joints).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='min', factor=0.5, patience=10, verbose=False)
    loss_fn   = nn.MSELoss()

    best_loss    = float('inf')
    patience_ctr = 0
    best_weights = None
    lossgraph    = []

    print("Retraining on variable-length windows...")
    for epoch in range(epochs):
        total_loss = 0
        for padded, lengths in loader:
            padded   = padded.to(device)
            recon, _ = model(padded, lengths=lengths)
            loss = torch.tensor(0.0, device=device)
            for i, l in enumerate(lengths):
                loss = loss + loss_fn(recon[i, :l], padded[i, :l])
            loss = loss / len(lengths)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        avg = total_loss / len(loader)
        lossgraph.append(avg)
        scheduler.step(avg)

        if avg < best_loss - 1e-4:
            best_loss    = avg
            patience_ctr = 0
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1

        if patience_ctr >= patience:
            print(f"Early stopping at epoch {epoch+1} — best loss: {best_loss:.6f}")
            model.load_state_dict(best_weights)
            break

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} — Loss: {avg:.6f}  "
                  f"patience: {patience_ctr}/{patience}")

    return model, lossgraph

# ============================================================
# STAGE 2: Compute reconstruction loss signal
# ============================================================

def compute_frame_loss(model, raw_sequence, window_size,
                        stride=5, device=device):
    model.eval()
    losses    = []
    positions = []

    raw_tensor = torch.tensor(raw_sequence, dtype=torch.float32, device=device)
    loss_fn    = nn.MSELoss()

    with torch.no_grad():
        for start in range(0, len(raw_sequence) - window_size, stride):
            window   = raw_tensor[start:start + window_size].unsqueeze(0)
            recon, _ = model(window)
            loss     = loss_fn(recon, window).item()
            losses.append(loss)
            positions.append(start + window_size // 2)

            if (start) % 100000 == 0:
                print(start,end="|")

    positions = np.array(positions)
    losses    = np.array(losses)
    print(f"Computed loss at {len(losses)} positions")
    print(f"Loss stats — mean: {losses.mean():.4f}, "
          f"std: {losses.std():.4f}, max: {losses.max():.4f}")
    return positions, losses


# ============================================================
# STAGE 3: Detect transitions
# ============================================================

def find_transitions(positions, losses,
                      percentile=70, smoothing=5,
                      min_distance=3, fps=30):
    if len(losses) < smoothing:
        smoothing = max(3, len(losses) // 2)
        if smoothing % 2 == 0:
            smoothing += 1

    smoothed  = savgol_filter(losses, window_length=smoothing, polyorder=2)
    threshold = np.percentile(smoothed, percentile)
    peaks, _  = find_peaks(smoothed, height=threshold, distance=min_distance)
    transition_frames = positions[peaks]

    if len(transition_frames) > 1:
        intervals = np.diff(transition_frames)
        mean_bout = np.mean(intervals) / fps
        print(f"Found {len(transition_frames)} transitions")
        print(f"Mean bout duration: {mean_bout:.2f}s")
        if mean_bout < 1:
            print("WARNING: bouts too short — raise percentile or min_distance")
        elif mean_bout > 15:
            print("WARNING: bouts too long — lower percentile")
        else:
            print("Bout duration looks plausible")

    return transition_frames, smoothed


# ============================================================
# STAGE 4: Variable-length windows from segments
# ============================================================

def create_windows_from_transitions(raw_sequence, transition_frames,
                                     min_segment_frames=15,
                                     max_segment_frames=300):
    n_frames   = len(raw_sequence)
    boundaries = np.unique(
        np.concatenate([[0], transition_frames, [n_frames]])
    ).astype(int)

    all_windows   = []
    window_labels = []

    for seg_idx in range(len(boundaries) - 1):
        seg_start = boundaries[seg_idx]
        seg_end   = boundaries[seg_idx + 1]
        seg_len   = seg_end - seg_start

        if seg_len < min_segment_frames:
            continue

        segment = raw_sequence[seg_start:seg_end]

        for start in range(0, seg_len, max_segment_frames):
            chunk = segment[start:start + max_segment_frames]
            if len(chunk) < min_segment_frames:
                continue
            all_windows.append(chunk)
            window_labels.append(seg_idx)

    lengths = [len(w) for w in all_windows]
    print(f"Created {len(all_windows)} variable-length windows from "
          f"{len(boundaries) - 1} segments")
    print(f"Window lengths — min: {min(lengths)}, "
          f"max: {max(lengths)}, mean: {np.mean(lengths):.1f}")

    return all_windows, np.array(window_labels)


# ============================================================
# STAGE 5: Encode, UMAP, cluster
# ============================================================

def encode_and_cluster(model, windows, batch_size=BATCH_SIZE,
                        device=device, quantile=0.1,
                        umap_neighbors=30, umap_min_dist=0.1):
    model.eval()
    all_latents = []

    tensor_list = [torch.tensor(w, dtype=torch.float32) for w in windows]
    loader = DataLoader(tensor_list, batch_size=batch_size, shuffle=False,
                        collate_fn=collate_variable_length)

    with torch.no_grad():
        for padded, lengths in loader:
            padded = padded.to(device)
            _, z   = model(padded, lengths=lengths)
            all_latents.append(z.cpu().numpy())

    all_latents = np.concatenate(all_latents, axis=0)  # (N, 16)
    print(f"Latents shape: {all_latents.shape}")

    # UMAP: 16D → 2D
    print("Running UMAP...")
    reducer    = UMAP(n_components=2, n_neighbors=umap_neighbors,
                      min_dist=umap_min_dist, metric='cosine',
                      random_state=42)
    latents_2d = reducer.fit_transform(all_latents)     # (N, 2)
    print(f"UMAP done. Shape: {latents_2d.shape}")

    # MeanShift on 2D UMAP embedding
    normed    = normalize(latents_2d, norm='l2')
    bandwidth = estimate_bandwidth(normed, quantile=quantile)
    print(f"Estimated bandwidth: {bandwidth:.4f}")

    ms     = MeanShift(bandwidth=bandwidth, bin_seeding=True)
    ms.fit(normed)
    labels = ms.labels_

    n_clusters = len(np.unique(labels))
    print(f"Found {n_clusters} clusters")
    print(f"Cluster sizes: {np.bincount(labels)}")

    if n_clusters > 1:
        dist_matrix = np.clip(1 - np.dot(normed, normed.T), 0, 2)
        sil = silhouette_score(dist_matrix, labels, metric='precomputed')
        print(f"Silhouette score: {sil:.4f}")

    return all_latents, latents_2d, labels, ms


# ============================================================
# FULL TWO-PASS PIPELINE
# ============================================================

def run_pipeline_test(raw_sequence, window_size, fps,
                      epochs, percentile, quantile,
                      min_segment_frames, max_segment_frames,
                      stride=5, umap_neighbors=30,
                      umap_min_dist=0.1, device=device):

    # ── PASS 1: fixed windows → transition detection ───────────────────
    print("\n" + "="*50)
    print("STAGE 1: Training RAE on fixed windows")
    print("="*50)
    n_joints  = raw_sequence.shape[1]
    joint_dim = raw_sequence.shape[2]

    model_stage1 = HierarchicalRAE(latent_dim=30, joint_dim=joint_dim,
                                    num_joints=n_joints).to(device)
    state_dict = torch.load('model_stage1.pth', weights_only=True)
    model_stage1.load_state_dict(state_dict)
    print("model_stage_1 loaded...")

    print("\n" + "="*50)
    print("STAGE 2: Computing reconstruction loss signal")
    print("="*50)
    positions = np.load('fix_win_positions.npy')
    losses = np.load('fix_win_losses.npy')
    print("position and loss loaded...")

    print("\n" + "="*50)
    print("STAGE 3: Finding behavioral transitions")
    print("="*50)
    transition_frames, smoothed = find_transitions(
        positions, losses,
        percentile=percentile, fps=fps
    )

    print("\n" + "="*50)
    print("STAGE 4: Creating variable-length behavioral windows")
    print("="*50)
    windows, window_segment_labels = create_windows_from_transitions(
        raw_sequence, transition_frames,
        min_segment_frames=min_segment_frames,
        max_segment_frames=max_segment_frames
    )

    # ── PASS 2: retrain on variable-length windows ─────────────────────
    print("\n" + "="*50)
    print("STAGE 4b: Retraining RAE on variable-length windows")
    print("="*50)
    model_stage2, lossgraph_stage2 = train_on_variable_windows(
        raw_sequence, windows,
        epochs=epochs, lr=LR, device=device
    )

    print("\n" + "="*50)
    print("STAGE 5: Encoding + UMAP + clustering")
    print("="*50)
    latents, latents_2d, cluster_labels, ms_model = encode_and_cluster(
        model_stage2, windows,
        quantile=quantile,
        umap_neighbors=umap_neighbors,
        umap_min_dist=umap_min_dist,
        device=device
    )

    return {
        'model':                 model_stage2,
        'model_stage1':          model_stage1,
        'latents':               latents,        # (N, 16) — raw high-dim latents
        'latents_2d':            latents_2d,     # (N, 2)  — UMAP projection
        'cluster_labels':        cluster_labels,
        'transition_frames':     transition_frames,
        'windows':               windows,
        'window_segment_labels': window_segment_labels,
        'losses':                losses,
        'positions':             positions,
        'lossgraph_stage1':      lossgraph_stage1,
        'lossgraph_stage2':      lossgraph_stage2,
        'smoothed_losses':       smoothed,
    }

In [9]:
# ==============================================================================
# HYPERPARAMETER TUNING: Bayesian optimization over percentile, quantile, (& lr)
# ==============================================================================

# ── Train Stage 1 ONCE (shared across all iterations) ────────
print("="*60)
print("PRE-STEP: Training Stage 1 model (shared across iterations)")
print("="*60)
model_stage1, lossgraph_stage1 = train_on_fixed_windows(
    raw_processed, window_size=WINDOW_SIZE, epochs=EPOCHS
)

torch.save(model_stage1.state_dict(), os.path.join('./', "model_stage1.pth"))


PRE-STEP: Training Stage 1 model (shared across iterations)
Extracted 125243 non-overlapping windows
Coverage: 3757290/3757302 frames (100.0%)


c:\Users\fbai_\anaconda3\envs\condaenv312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training on fixed windows...
Epoch 5/2000 — Loss: 0.573306  patience: 0/30
Epoch 10/2000 — Loss: 0.514861  patience: 0/30
Epoch 15/2000 — Loss: 0.468069  patience: 0/30
Epoch 20/2000 — Loss: 0.449282  patience: 0/30
Epoch 25/2000 — Loss: 0.434910  patience: 1/30
Epoch 30/2000 — Loss: 0.418643  patience: 1/30
Epoch 35/2000 — Loss: 0.401492  patience: 0/30
Epoch 40/2000 — Loss: 0.392227  patience: 0/30
Epoch 45/2000 — Loss: 0.385924  patience: 2/30
Epoch 50/2000 — Loss: 0.375022  patience: 2/30
Epoch 55/2000 — Loss: 0.364830  patience: 0/30
Epoch 60/2000 — Loss: 0.360757  patience: 3/30
Epoch 65/2000 — Loss: 0.349181  patience: 0/30
Epoch 70/2000 — Loss: 0.345005  patience: 1/30
Epoch 75/2000 — Loss: 0.339880  patience: 1/30
Epoch 80/2000 — Loss: 0.334680  patience: 0/30
Epoch 85/2000 — Loss: 0.324588  patience: 0/30
Epoch 90/2000 — Loss: 0.323927  patience: 0/30
Epoch 95/2000 — Loss: 0.316949  patience: 2/30
Epoch 100/2000 — Loss: 0.319142  patience: 3/30
Epoch 105/2000 — Loss: 0.315582

In [10]:
print("\n" + "="*60)
print("PRE-STEP: Computing reconstruction loss signal (shared)")
print("="*60)
positions, losses = compute_frame_loss(
    model_stage1, raw_processed, window_size=WINDOW_SIZE, stride=5, device=device
)

np.save('fix_win_positions.npy', positions)
np.save('fix_win_losses.npy', losses)


PRE-STEP: Computing reconstruction loss signal (shared)
0|100000|200000|300000|400000|500000|600000|700000|800000|900000|1000000|1100000|1200000|1300000|1400000|1500000|1600000|1700000|1800000|1900000|2000000|2100000|2200000|2300000|2400000|2500000|2600000|2700000|2800000|2900000|3000000|3100000|3200000|3300000|3400000|3500000|3600000|3700000|Computed loss at 751455 positions
Loss stats — mean: 0.3295, std: 2.1778, max: 385.1589


In [ ]:
with open("lossgraph_stage1.json", "w") as file:
    json.dump(lossgraph_stage1, file)

Load posistions, losses and model_stage_1 from local disk

In [11]:
positions = np.load('fix_win_positions.npy')
losses = np.load('fix_win_losses.npy')

In [12]:
n_joints  = raw_processed.shape[1]
joint_dim = raw_processed.shape[2]

model_stage1 = HierarchicalRAE(latent_dim=30, joint_dim=joint_dim,
                                num_joints=n_joints).to(device)
state_dict = torch.load('model_stage1.pth', weights_only=True)
model_stage1.load_state_dict(state_dict)
#model_stage1.eval()

with open("lossgraph_stage1.json", "r") as file:
    lossgraph_stage1 = json.load(file)

Bayesian search for optimal percentile and quantile 

In [13]:
MAX_ITER = 15
EPOCHS = 2000
VW_EPOCHS = 1500
BATCH_SIZE = 256     #640   # 1280    # 1536  # 128X12
LR = 1e-3
WINDOW_SIZE = 60

PERCENTILE_RANGE = (50,75)   #(45.0, 70.0)
QUANTILE_RANGE = (0.05,0.20)   # (0.05, 0.2)
LR_RANGE = (1e-4, 1e-2)

In [14]:
# ── Define search space ──────────────────────────────────────
search_space = [
    Real(*PERCENTILE_RANGE, name='percentile'),
    Real(*QUANTILE_RANGE,   name='quantile'),
    #Real(*LR_RANGE,         name='lr', prior='log-uniform'),
]

search_log = []
best_results = None
best_score = -1
iteration = 0

In [15]:
# ── Objective function ───────────────────────────────────────
@use_named_args(search_space)
def objective(percentile, quantile, lr=LR):
    global iteration, best_score, best_results

    iteration += 1
    print(f"\n{'='*60}")
    print(f"ITERATION {iteration}/{MAX_ITER}  |  percentile={percentile:.2f}, quantile={quantile:.4f}, lr={lr:.6f}")
    print(f"{'='*60}")

    # Stage 3: transitions
    transition_frames, smoothed = find_transitions(
        positions, losses, percentile=percentile, fps=30
    )

    # Stage 4: variable windows
    windows, window_segment_labels = create_windows_from_transitions(
        raw_processed, transition_frames,
        min_segment_frames=60, max_segment_frames=600
    )

    # Stage 4b: retrain on variable windows (with tuned lr)
    model_file_name = "model_stage2_"+str(iteration)+".pth"
    if Path(model_file_name).is_file():
        n_joints  = raw_processed.shape[1]
        joint_dim = raw_processed.shape[2]

        model_stage2 = HierarchicalRAE(latent_dim=30, joint_dim=joint_dim,
                                num_joints=n_joints).to(device)
        state_dict = torch.load(model_file_name, weights_only=True)
        model_stage2.load_state_dict(state_dict)
        print("model_satge2 loaded...")
    else:
        model_stage2, lossgraph_stage2 = train_on_variable_windows(
            raw_processed, windows, epochs=VW_EPOCHS, lr=lr, device=device
        )
        torch.save(model_stage2.state_dict(), os.path.join('./', model_file_name))

    # Stage 5: encode + cluster
    latents, latents_2d, cluster_labels, ms_model = encode_and_cluster(
        model_stage2, windows, quantile=quantile,
        umap_neighbors=30, umap_min_dist=0.1, device=device
    )

    n_clusters = len(np.unique(cluster_labels))

    # ── Compute silhouette score ──────────────────────────────
    # silhouette needs at least 2 clusters and more samples than clusters
    if n_clusters < 2 or n_clusters >= len(latents_2d):
        print(f"  → {n_clusters} clusters — skipping silhouette (invalid cluster count)")
        score = -1.0
    else:
        score = silhouette_score(latents_2d, cluster_labels)

    log_entry = {
        "iteration": iteration,
        "percentile": round(percentile, 2),
        "quantile": round(quantile, 4),
        "lr": round(float(lr), 6),
        "n_clusters": n_clusters,
        "silhouette": round(float(score), 4),
    }
    search_log.append(log_entry)
    print(f"  → {n_clusters} clusters, silhouette={score:.4f}")

    # Track best
    if score > best_score:
        best_score = score
        best_results = {
            'model': model_stage2,
            'model_stage1': model_stage1,
            'latents': latents,
            'latents_2d': latents_2d,
            'cluster_labels': cluster_labels,
            'transition_frames': transition_frames,
            'windows': windows,
            'window_segment_labels': window_segment_labels,
            'losses': losses,
            'positions': positions,
            'lossgraph_stage1': lossgraph_stage1,
            #'lossgraph_stage2': lossgraph_stage2,
            'smoothed_losses': smoothed,
            '_percentile': percentile,
            '_quantile': quantile,
            '_lr': lr,
        }
        print(f"  ** NEW BEST (silhouette={score:.4f}) **")

    # Cleanup GPU memory
    del model_stage2, latents, latents_2d, cluster_labels, ms_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # gp_minimize minimizes, so return negative score
    return -score

In [16]:
# ── Run Bayesian optimization ────────────────────────────────
bayes_result = gp_minimize(
    func=objective,
    dimensions=search_space,
    n_calls=MAX_ITER,
    n_initial_points=min(MAX_ITER, 5),
    random_state=42,
    verbose=False,
)


ITERATION 1/15  |  percentile=69.91, quantile=0.0775, lr=0.001000
Found 39005 transitions
Mean bout duration: 3.21s
Bout duration looks plausible
Created 14029 variable-length windows from 39006 segments
Window lengths — min: 60, max: 600, mean: 211.6
model_satge2 loaded...
Latents shape: (14029, 30)
Running UMAP...


c:\Users\fbai_\anaconda3\envs\condaenv312\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP done. Shape: (14029, 2)
Estimated bandwidth: 0.2124
Found 11 clusters
Cluster sizes: [1935 2473 2044 1894 1509 1622  765  432  488  395  472]
Silhouette score: 0.6791
  → 11 clusters, silhouette=0.1789
  ** NEW BEST (silhouette=0.1789) **

ITERATION 2/15  |  percentile=69.49, quantile=0.1395, lr=0.001000
Found 39489 transitions
Mean bout duration: 3.17s
Bout duration looks plausible
Created 14060 variable-length windows from 39490 segments
Window lengths — min: 60, max: 600, mean: 210.2
model_satge2 loaded...
Latents shape: (14060, 30)
Running UMAP...


c:\Users\fbai_\anaconda3\envs\condaenv312\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP done. Shape: (14060, 2)
Estimated bandwidth: 0.3873
Found 3 clusters
Cluster sizes: [5889 5981 2190]
Silhouette score: 0.7151
  → 3 clusters, silhouette=0.4118
  ** NEW BEST (silhouette=0.4118) **

ITERATION 3/15  |  percentile=61.15, quantile=0.0650, lr=0.001000
Found 48351 transitions
Mean bout duration: 2.59s
Bout duration looks plausible
Created 14870 variable-length windows from 48352 segments
Window lengths — min: 60, max: 600, mean: 183.0
model_satge2 loaded...
Latents shape: (14870, 30)
Running UMAP...


...

UMAP done. Shape: (14101, 2)
Estimated bandwidth: 0.1227
Found 8 clusters
Cluster sizes: [5910 2664 2065 1494 1679  218   59   12]
Silhouette score: 0.6856
  → 8 clusters, silhouette=0.1132

ITERATION 14/15  |  percentile=68.95, quantile=0.1421, lr=0.001000
Found 40018 transitions
Mean bout duration: 3.13s
Bout duration looks plausible
Created 14121 variable-length windows from 40019 segments
Window lengths — min: 60, max: 600, mean: 208.3
model_satge2 loaded...
Latents shape: (14121, 30)
Running UMAP...


c:\Users\fbai_\anaconda3\envs\condaenv312\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP done. Shape: (14121, 2)
Estimated bandwidth: 0.0733
Found 7 clusters
Cluster sizes: [4060 5706 2888 1365   58   30   14]
Silhouette score: 0.6777
  → 7 clusters, silhouette=0.2491

ITERATION 15/15  |  percentile=51.33, quantile=0.1493, lr=0.001000
Found 58710 transitions
Mean bout duration: 2.13s
Bout duration looks plausible
Created 15457 variable-length windows from 58711 segments
Window lengths — min: 60, max: 600, mean: 157.5
model_satge2 loaded...
Latents shape: (15457, 30)
Running UMAP...


c:\Users\fbai_\anaconda3\envs\condaenv312\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP done. Shape: (15457, 2)
Estimated bandwidth: 0.1368
Found 4 clusters
Cluster sizes: [6622 6812 1572  451]
Silhouette score: 0.7753
  → 4 clusters, silhouette=0.4154
  ** NEW BEST (silhouette=0.4154) **


In [17]:
# ── Summary ──────────────────────────────────────────────────
print(f"\n{'='*60}")
print("HYPERPARAMETER SEARCH COMPLETE")
print(f"{'='*60}")
print(f"\nSearch log:")
for entry in search_log:
    marker = " <-- BEST" if entry['silhouette'] == best_score else ""
    print(f"  Iter {entry['iteration']:2d}: percentile={entry['percentile']:5.2f}, "
          f"quantile={entry['quantile']:.4f}, lr={entry['lr']:.6f}  "
          f"→  {entry['n_clusters']} clusters, silhouette={entry['silhouette']:.4f}{marker}")

print(f"\nBest result:")
print(f"  percentile : {best_results['_percentile']:.2f}")
print(f"  quantile   : {best_results['_quantile']:.4f}")
print(f"  lr         : {best_results['_lr']:.6f}")
print(f"  clusters   : {len(np.unique(best_results['cluster_labels']))}")
print(f"  silhouette : {best_score:.4f}")

# Set results to best for downstream cells
results = best_results

# Save the model
torch.save(results['model'].state_dict(), os.path.join('./', "model_stage2.pth"))

with open("lossgraph_stage2.json", "w") as file:
    json.dump(results['lossgraph_stage2'], file)


HYPERPARAMETER SEARCH COMPLETE

Search log:
  Iter  1: percentile=69.91, quantile=0.0775, lr=0.001000  →  11 clusters, silhouette=0.1789
  Iter  2: percentile=69.49, quantile=0.1395, lr=0.001000  →  3 clusters, silhouette=0.4118
  Iter  3: percentile=61.15, quantile=0.0650, lr=0.001000  →  11 clusters, silhouette=0.1915
  Iter  4: percentile=61.48, quantile=0.1001, lr=0.001000  →  7 clusters, silhouette=0.2624
  Iter  5: percentile=53.57, quantile=0.1476, lr=0.001000  →  5 clusters, silhouette=0.3079
  Iter  6: percentile=74.53, quantile=0.1407, lr=0.001000  →  6 clusters, silhouette=0.2179
  Iter  7: percentile=51.96, quantile=0.1413, lr=0.001000  →  5 clusters, silhouette=0.2294
  Iter  8: percentile=69.62, quantile=0.1392, lr=0.001000  →  6 clusters, silhouette=0.2775
  Iter  9: percentile=69.45, quantile=0.1396, lr=0.001000  →  4 clusters, silhouette=0.2933
  Iter 10: percentile=69.44, quantile=0.1397, lr=0.001000  →  4 clusters, silhouette=0.2815
  Iter 11: percentile=68.24, quan

KeyError: 'lossgraph_stage2'

Run optimal model_stage_2 with the optimal parameters

In [ ]:
"""
percentile=67.69 
quantile=0.1282

results = run_pipeline_test(
    raw_processed, 
    window_size=WINDOW_SIZE,
    fps=30,
    epochs=EPOCHS,
    percentile=percentile,
    quantile=quantile,
    min_segment_frames=60, 
    max_segment_frames=600,
    stride = 5,
    device=device
)

# Save the model
torch.save(results['model'].state_dict(), os.path.join('./', "model_stage2.pth"))
"""

In [ ]:
import matplotlib.colors as mcolors
embedded = results['latents']
labels = results['cluster_labels']
plt.figure(figsize=(8, 6))
plt.scatter(embedded[:, 0], embedded[:, 1])
plt.show()


"""latents = results['latents']
normed  = normalize(latents, norm='l2')
labels = results['cluster_labels']
# UMAP to 2D
reducer   = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.5, random_state=42)
embedded  = reducer.fit_transform(normed)
"""
embedded = results['latents']
labels = results['cluster_labels']

unique_labels = np.sort(np.unique(labels))
n_clusters = len(unique_labels)

cmap = plt.get_cmap('tab10', n_clusters)
norm = mcolors.BoundaryNorm(np.arange(n_clusters + 1) - 0.5, n_clusters)

plt.figure(figsize=(8, 6))
sc = plt.scatter(embedded[:, 0], embedded[:, 1],
                  c=labels, cmap=cmap, norm=norm, s=15, alpha=0.8)

cbar = plt.colorbar(sc, label='Cluster', ticks=unique_labels)
cbar.ax.set_yticklabels([str(int(l)) for l in unique_labels])

plt.show()

In [ ]:
import cv2
import json
import numpy as np
from pathlib import Path
from collections import defaultdict

# ── settings ───────────────────────────────────────────────────────────────
json_folder   = Path(r"E:\ferg_data\json_files")
video_folder  = Path(r"E:\ferg_data\videos")
output_folder = Path(r"E:\ferg_data\reconstructed_vids")
output_folder.mkdir(exist_ok=True)

n_cols             = 2
n_rows             = 2
n_examples         = n_cols * n_rows   # 12
fps_out            = 30
cell_w, cell_h     = 320, 240
min_segment_frames = 60
max_segment_frames = 600

# ── step 1: load json files in the SAME order used to build da ────────────
# (must match json_folder.glob("*.json") exactly, same as the da-building
# cell — no manual ordering/exclusion here, since glob() only returns files
# that actually exist on disk)
json_names_ordered = [f.name for f in json_folder.glob("*.json")]
print(f"Found {len(json_names_ordered)} json files (glob order)")

print("Loading JSON files in glob order...")
json_names   = []
frame_counts = []

for json_name in json_names_ordered:
    json_file = json_folder / json_name
    if not json_file.exists():
        print(f"  ✗ {json_name} — file not found, skipping")
        continue
    with open(json_file) as f:
        data = json.load(f)
    n = len(data["annotations"])
    json_names.append(json_name)
    frame_counts.append(n)
    print(f"  {json_name}: {n} frames")

# keep json_names_ordered in sync with what actually loaded, since later
# steps (frame ID maps, verification) iterate over json_names_ordered
json_names_ordered = json_names

print(f"Total frames: {sum(frame_counts)}")
assert sum(frame_counts) == len(da), \
    f"Frame count mismatch: {sum(frame_counts)} vs {len(da)}"
print("Frame counts match da ✓")

# ── step 2: build json → actual video frame ID mapping ────────────────────
print("\nBuilding frame ID maps...")
json_frame_maps = {}
for json_name in json_names_ordered:
    json_file = json_folder / json_name
    with open(json_file) as f:
        data = json.load(f)
    frame_ids = list(data["annotations"].keys())
    json_frame_maps[json_name] = {
        i: int(fid) for i, fid in enumerate(frame_ids)
    }
    print(f"  {json_name}: frames {frame_ids[0]} → {frame_ids[-1]}")

# ── step 3: build global frame → (json_name, local_annotation_idx) ────────
print("\nBuilding frame_to_video mapping...")
frame_to_video = {}
cumulative     = 0
for name, count in zip(json_names, frame_counts):
    for local_frame in range(count):
        frame_to_video[cumulative + local_frame] = (name, local_frame)
    cumulative += count
print(f"frame_to_video built: {len(frame_to_video)} entries")

# ── verify mapping is correct ──────────────────────────────────────────────
json_name_0, local_idx_0 = frame_to_video[0]
actual_vid_0 = json_frame_maps[json_name_0].get(local_idx_0, local_idx_0)
json_file_0  = json_folder / json_name_0
with open(json_file_0) as f:
    d0 = json.load(f)
frame_ids_0 = list(d0["annotations"].keys())
first_kps   = list(d0["annotations"][frame_ids_0[0]].values())
first_x     = [kp[0] for kp in first_kps]
da_x        = da.values[0, :, 0].tolist()
match       = np.allclose(first_x, da_x, atol=1)
print(f"\nVerification: frame_to_video[0] → {json_name_0}")
print(f"  json x: {first_x}")
print(f"  da x:   {da_x}")
print(f"  Match: {match} {'✓' if match else '✗ — ORDER IS WRONG'}")

# extra check partway through the sequence, not just frame 0 — catches
# ordering drift that a frame-0-only check would miss
mid_idx = len(json_names) // 2
mid_json = json_names[mid_idx]
mid_global_start = sum(frame_counts[:mid_idx])
mid_json_check, _ = frame_to_video[mid_global_start]
print(f"Mid-sequence check: frame_to_video[{mid_global_start}] → "
      f"{mid_json_check} (expected {mid_json}) "
      f"{'✓' if mid_json_check == mid_json else '✗ — ORDER IS WRONG'}")

# ── step 4: find matching video files ─────────────────────────────────────
print("\nSearching for video files...")
all_videos    = list(video_folder.rglob("*.avi"))
video_by_name = {v.stem: v for v in all_videos}

video_map = {}
for json_name in json_names:
    stem = json_name.replace('_predictions.json', '')
    if stem in video_by_name:
        video_map[json_name] = video_by_name[stem]
        print(f"  ✓ {json_name}")
    else:
        print(f"  ✗ {json_name} — no matching video")
print(f"video_map has {len(video_map)} entries")

In [ ]:
# ── step 5: reconstruct window start frames ────────────────────────────────
print("\nReconstructing window starts...")
n_frames_total = len(da)
boundaries     = np.unique(
    np.concatenate([[0], results['transition_frames'], [n_frames_total]])
).astype(int)

starts = []
for seg_idx in range(len(boundaries) - 1):
    seg_start = boundaries[seg_idx]
    seg_end   = boundaries[seg_idx + 1]
    seg_len   = seg_end - seg_start
    if seg_len < min_segment_frames:
        continue
    for chunk_start in range(0, seg_len, max_segment_frames):
        abs_start = seg_start + chunk_start
        abs_end   = min(abs_start + max_segment_frames, seg_end)
        if (abs_end - abs_start) < min_segment_frames:
            continue
        starts.append(abs_start)

starts = np.array(starts)
print(f"Window starts: {len(starts)} vs windows: {len(results['windows'])}")
assert len(starts) == len(results['windows']), \
    f"Mismatch: {len(starts)} starts vs {len(results['windows'])} windows"

# ── step 6: skeleton and helper functions ──────────────────────────────────
SKELETON = [(0,1),(1,2),(1,3),(1,4),(2,5),(2,6)]

def draw_skeleton(frame, keypoints, color=(0,255,0), thickness=2):
    kp = keypoints.astype(int)
    for (a, b) in SKELETON:
        cv2.line(frame, tuple(kp[a]), tuple(kp[b]), color, thickness)
    for (x, y) in kp:
        cv2.circle(frame, (x, y), 4, color, -1)
    return frame

_cap_cache = {}
_cap_pos   = {}

def get_frame_sequential(json_name, local_frm):
    if json_name not in video_map:
        return None
    actual_frm = json_frame_maps[json_name].get(local_frm, local_frm)

    if json_name not in _cap_cache:
        _cap_cache[json_name] = cv2.VideoCapture(str(video_map[json_name]))
        _cap_pos[json_name]   = 0

    cap = _cap_cache[json_name]
    cur = _cap_pos[json_name]

    if actual_frm < cur:
        cap.release()
        _cap_cache[json_name] = cv2.VideoCapture(str(video_map[json_name]))
        _cap_pos[json_name]   = 0
        cap = _cap_cache[json_name]
        cur = 0

    while cur < actual_frm:
        cap.read()
        cur += 1

    ret, frame = cap.read()
    cur += 1
    _cap_pos[json_name] = cur
    return frame if ret else None

def close_cap_cache():
    for cap in _cap_cache.values():
        cap.release()
    _cap_cache.clear()
    _cap_pos.clear()

def window_has_video(win_idx):
    window    = results['windows'][win_idx]
    abs_start = starts[win_idx]
    for local_t in range(min(5, len(window))):
        global_frame  = abs_start + local_t
        json_name, _  = frame_to_video[global_frame]
        if json_name not in video_map:
            return False
    return True

def load_window_frames(win_idx):
    window    = results['windows'][win_idx]
    abs_start = starts[win_idx]
    frames    = []
    for local_t in range(len(window)):
        global_frame         = abs_start + local_t
        json_name, local_frm = frame_to_video[global_frame]
        frame = get_frame_sequential(json_name, local_frm)
        if frame is not None:
            raw_kp = da.values[global_frame]
            frame  = draw_skeleton(frame, raw_kp, color=(0, 255, 0))
            frame  = cv2.resize(frame, (cell_w, cell_h))
        else:
            frame  = np.zeros((cell_h, cell_w, 3), dtype=np.uint8)
        frames.append(frame)
    return frames

# ── step 7: create one grid video per cluster ──────────────────────────────
n_clusters = len(np.unique(results['cluster_labels']))
latents_2d = results['latents_2d']
labels     = results['cluster_labels']

print(f"\nCreating grid videos for {n_clusters} clusters...")

for c in range(n_clusters):
    print(f"\nCluster {c}...")

    cluster_idx     = np.where(labels == c)[0]
    cluster_latents = latents_2d[cluster_idx]
    centroid        = cluster_latents.mean(axis=0)
    dists           = np.linalg.norm(cluster_latents - centroid, axis=1)
    closest_order   = np.argsort(dists)

    chosen = []
    for idx in cluster_idx[closest_order]:
        if window_has_video(idx):
            chosen.append(idx)
        if len(chosen) == n_examples:
            break
    chosen = np.array(chosen)
    print(f"  {len(cluster_idx)} windows total, {len(chosen)} with valid video")

    if len(chosen) == 0:
        print(f"  Skipping cluster {c} — no valid videos")
        continue

    close_cap_cache()

    print(f"  Loading frames...")
    all_window_frames = []
    for i, win_idx in enumerate(chosen):
        print(f"    Window {i+1}/{len(chosen)} "
              f"(idx={win_idx}, len={len(results['windows'][win_idx])})")
        wf = load_window_frames(win_idx)
        all_window_frames.append(wf)

    max_len  = max(len(wf) for wf in all_window_frames)
    grid_w   = cell_w * n_cols
    grid_h   = cell_h * n_rows + 40
    out_path = output_folder / f"cluster_{c}_grid.mp4"
    fourcc   = cv2.VideoWriter_fourcc(*'mp4v')
    out      = cv2.VideoWriter(str(out_path), fourcc, fps_out, (grid_w, grid_h))
    blank    = np.zeros((cell_h, cell_w, 3), dtype=np.uint8)

    print(f"  Writing {max_len} frames to {out_path.name}...")
    for t in range(max_len):
        grid = np.zeros((grid_h, grid_w, 3), dtype=np.uint8)
        cv2.putText(grid, f"Cluster {c}  |  Frame {t+1}/{max_len}",
                    (10, 28), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (255, 255, 255), 2)

        for i in range(n_examples):
            row     = i // n_cols
            col     = i % n_cols
            y_start = 40 + row * cell_h
            x_start = col * cell_w

            if i < len(all_window_frames):
                wf    = all_window_frames[i]
                frame = wf[t] if t < len(wf) else blank.copy()
            else:
                frame = blank.copy()

            if frame is None:
                frame = blank.copy()

            cv2.putText(frame, f"W{chosen[i]}",
                        (5, 20), cv2.FONT_HERSHEY_SIMPLEX,
                        0.5, (0, 255, 0), 1)

            grid[y_start:y_start+cell_h, x_start:x_start+cell_w] = frame

        out.write(grid)

    out.release()
    print(f"  Saved → {out_path}")

close_cap_cache()
print(f"\nDone! {n_clusters} videos saved to {output_folder}")

In [ ]:
# ============================================================
# Visual representation of the significant motifs — pose only
# Reuses: da, results, starts, frame_to_video, video_map, json_names_ordered
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

START_STOP = LinearSegmentedColormap.from_list(
    "start_stop", ["#2ab7e0", "#ffffff", "#d6418f"])

SKELETON    = [(0,1),(1,2),(1,3),(1,4),(2,5),(2,6)]
NOSE        = 0
MAX_FRAMES  = 80       # max poses to overlay per panel
ALPHA_SKEL  = 0.25     # skeleton stroke opacity
ALPHA_NOSE  = 0.9      # nose trajectory opacity
PAD         = 0.15     # axis padding (fraction)
MOTIF_NAMES = {}       # optional {0: "Turning", ...}

labels  = results['cluster_labels']
latents = results['latents']           # full-dim for accurate centroid distance


def window_has_video(w):
    """A window is usable only if every frame in it:
      1. maps to a json/video session (frame_to_video), and
      2. that session actually has a matching video file (video_map), and
      3. the window doesn't straddle two different sessions (which would
         make the pose sequence jump between unrelated videos).
    """
    abs_start = starts[w]
    L         = len(results['windows'][w])
    abs_end   = abs_start + L - 1

    if abs_start not in frame_to_video or abs_end not in frame_to_video:
        return False

    start_json, _ = frame_to_video[abs_start]
    end_json, _   = frame_to_video[abs_end]

    if start_json != end_json:
        return False  # window spans two sessions — skip it

    return start_json in video_map


def representative_window(c):
    idx = np.where(labels == c)[0]
    if len(idx) == 0:
        return None
    cen   = latents[idx].mean(axis=0)
    order = idx[np.argsort(np.linalg.norm(latents[idx] - cen, axis=1))]
    for w in order:
        if window_has_video(int(w)):
            return int(w)
    return None


def draw_motif_pose(ax, win_idx):
    abs_start = starts[win_idx]
    L         = len(results['windows'][win_idx])

    # evenly subsample up to MAX_FRAMES poses
    step = max(1, L // MAX_FRAMES)
    kp   = da.values[abs_start:abs_start + L:step][:MAX_FRAMES]  # (T, 7, 2)
    T    = len(kp)

    ax.set_facecolor("black")

    for i in range(T):
        t   = i / max(T - 1, 1)
        col = START_STOP(t)

        # skeleton limbs
        for j0, j1 in SKELETON:
            xs = [kp[i, j0, 0], kp[i, j1, 0]]
            ys = [kp[i, j0, 1], kp[i, j1, 1]]
            ax.plot(xs, ys, color=col, lw=1.8, alpha=ALPHA_SKEL,
                    solid_capstyle="round")

        # joint dots
        ax.scatter(kp[i, :, 0], kp[i, :, 1],
                   s=12, color=col, alpha=ALPHA_SKEL, zorder=3, edgecolors="none")

    # nose trajectory on top as white dotted line
    nose = kp[:, NOSE, :]
    ax.plot(nose[:, 0], nose[:, 1], ls=":", color="white", lw=1.4,
            marker="o", ms=2, mfc="white", mec="white", alpha=ALPHA_NOSE, zorder=5)

    # auto-scale with padding
    all_xy = kp.reshape(-1, 2)
    xmin, ymin = all_xy.min(0); xmax, ymax = all_xy.max(0)
    xpad = PAD * (xmax - xmin); ypad = PAD * (ymax - ymin)
    ax.set_xlim(xmin - xpad, xmax + xpad)
    ax.set_ylim(ymax + ypad, ymin - ypad)   # invert y (image coords)
    ax.set_xticks([]); ax.set_yticks([])


# ---- assemble figure ----
n_clusters = len(np.unique(labels))
panels = []
for c in range(n_clusters):
    w = representative_window(c)
    if w is not None:
        panels.append((c, w))
    else:
        print(f"  ✗ Cluster {c}: no window with a usable video found — skipping panel")

ncol = min(4, len(panels))
nrow = int(np.ceil(len(panels) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 3.4 * nrow), facecolor="white")
axes = np.atleast_1d(axes).ravel()
for ax in axes[len(panels):]:
    ax.axis("off")

for k, (ax, (c, w)) in enumerate(zip(axes, panels)):
    draw_motif_pose(ax, w)
    ax.text(0.03, 0.97, MOTIF_NAMES.get(c, f"Motif {c}"), transform=ax.transAxes,
            color="white", fontsize=11, va="top")
    ax.text(0.97, 0.97, str(c), transform=ax.transAxes, color="white",
            fontsize=20, fontweight="bold", va="top", ha="right")
    if k == 0:
        cax = ax.inset_axes([0.28, 0.04, 0.44, 0.05])
        cax.imshow(np.linspace(0, 1, 256)[None, :], aspect="auto", cmap=START_STOP)
        cax.set_xticks([]); cax.set_yticks([])
        ax.text(0.28, 0.11, "Start", transform=ax.transAxes, color="#2ab7e0", fontsize=9)
        ax.text(0.72, 0.11, "Stop",  transform=ax.transAxes, color="#d6418f",
                fontsize=9, ha="right")

fig.suptitle("Visual representation of the significant motifs", fontsize=14)
plt.tight_layout()
fig.savefig(output_folder / "significant_motifs_pose.png", dpi=200,
            bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# quick check before running:
for c in range(len(np.unique(results['cluster_labels']))):
    w = representative_window(c)
    if w is None: continue
    abs_start = starts[w]
    L = len(results['windows'][w])
    burst_start = abs_start + max(0, (L - 60) // 2)
    burst_end = burst_start + 60
    sess_start = frame_to_video[burst_start][0]
    sess_end   = frame_to_video[min(burst_end, len(frame_to_video)-1)][0]
    print(f"cluster {c}: window {w}, session {sess_start} -> {sess_end}, same={sess_start==sess_end}")

Save best_results to local disk

In [ ]:
with open("best_results.pkl", "wb") as file:
    pickle.dump(results, file)

Load best_results from local disk

In [ ]:
with open("best_results.pkl", "rb") as file:
    results = pickle.load(file)

#### Bayesian search of parameters with balance windows

In [ ]:
# ── Step 1: balance windows based on EXISTING cluster labels ───────────────
print("="*60)
print("STEP 1: Balancing windows based on existing clusters")
print("="*60)

cluster_labels_original = results['cluster_labels']
cluster_sizes_original  = np.bincount(cluster_labels_original)
print(f"Original cluster sizes: {cluster_sizes_original}")
print(f"Total windows: {len(cluster_labels_original)}")

target_size = int(np.median(cluster_sizes_original))
print(f"Target size per cluster: {target_size}")

np.random.seed(42)
balanced_indices = []
for c in np.unique(cluster_labels_original):
    idx    = np.where(cluster_labels_original == c)[0]
    chosen = np.random.choice(idx, min(target_size, len(idx)), replace=False)
    balanced_indices.extend(chosen)
    print(f"  Cluster {c}: {len(idx):5d} → {min(target_size, len(idx)):5d} windows")

balanced_indices = np.array(balanced_indices)
balanced_windows = [results['windows'][i] for i in balanced_indices]
print(f"\nBalanced dataset: {len(balanced_windows)} windows total")



In [ ]:
# ── Step 2: compute transition frames from existing results ────────────────
# these are reused inside the Bayesian loop for percentile tuning
positions        = results['positions']
losses           = results['losses']
all_windows_orig = results['windows']   # full unbalanced window set


In [ ]:
# ── Step 3: Bayesian optimization ─────────────────────────────────────────
print("\n" + "="*60)
print("STEP 3: Bayesian optimization over lr, quantile, percentile")
print("="*60)

MAX_ITER  = 10
VW_EPOCHS = 750

search_space = [
    Real(*PERCENTILE_RANGE, name='percentile'),
    Real(*QUANTILE_RANGE, name='quantile'),
    #Real(*LR_RANGE, name='lr', prior='log-uniform'),
]

new_search_log   = []
new_best_results = None
new_best_score   = -1
new_iteration    = 0


In [ ]:
@use_named_args(search_space)
def new_objective(percentile, quantile, lr=LR):
    global new_iteration, new_best_score, new_best_results

    new_iteration += 1
    print(f"\n{'='*60}")
    print(f"ITERATION {new_iteration}/{MAX_ITER} | percentile={percentile:.2f}, quantile={quantile:.4f}, lr={lr:.6f}")
    print(f"{'='*60}")

    # ── find new transitions with this percentile ──────────────────────────
    new_transition_frames, new_smoothed = find_transitions(
        positions, losses, percentile=percentile, fps=30
    )

    # ── create new variable windows with these transitions ─────────────────
    new_windows, new_window_segment_labels = create_windows_from_transitions(
        raw_processed, new_transition_frames,
        min_segment_frames=60, max_segment_frames=600
    )

    if len(new_windows) < 10:
        print(f"  → too few windows ({len(new_windows)}) — skipping")
        return 0.0

    # ── map balanced indices to new windows ────────────────────────────────
    # balanced_indices came from the original window set
    # we need to find the corresponding windows in the new set
    # use the balanced windows directly since they came from raw_processed
    # and are independent of transition detection
    train_windows = balanced_windows   # always train on balanced subset

    print(f"  Training on {len(train_windows)} balanced windows...")

    # ── retrain Stage 2 on balanced windows with this lr ──────────────────
    new_model_file_name = "new_model_stage2_"+str(new_iteration)+".pth"
    if Path(new_model_file_name).is_file():
        n_joints  = raw_processed.shape[1]
        joint_dim = raw_processed.shape[2]

        new_model_stage2 = HierarchicalRAE(latent_dim=30, joint_dim=joint_dim,
                                num_joints=n_joints).to(device)
        state_dict = torch.load(new_model_file_name, weights_only=True)
        new_model_stage2.load_state_dict(state_dict)
        print("new_model_satge2 loaded...")
    else:
        new_model_stage2, new_lossgraph_stage2 = train_on_variable_windows(
            raw_processed, train_windows, epochs = VW_EPOCHS, lr = lr, device = device
        )
        torch.save(new_model_stage2.state_dict(), os.path.join('./', new_model_file_name))

    # ── encode ALL new windows with trained model ──────────────────────────
    print(f"  Encoding {len(new_windows)} windows...")
    new_latents, new_latents_2d, new_cluster_labels, new_ms_model = encode_and_cluster(
        new_model_stage2, new_windows,   # ALL new windows
        quantile = quantile, umap_neighbors = 30, umap_min_dist = 0.1, device = device
    )

    new_n_clusters = len(np.unique(new_cluster_labels))

    # ── silhouette score ───────────────────────────────────────────────────
    if new_n_clusters < 2 or new_n_clusters >= len(new_latents_2d):
        print(f"  → {new_n_clusters} clusters — invalid")
        new_score = -1.0
    else:
        normed      = normalize(new_latents_2d, norm='l2')
        dist_matrix = np.clip(1 - np.dot(normed, normed.T), 0, 2)
        new_score       = silhouette_score(dist_matrix, new_cluster_labels,
                                        metric='precomputed')

    new_log_entry = {
        "iteration":  new_iteration,
        "percentile": round(percentile, 2),
        "quantile":   round(quantile, 4),
        "lr":         round(float(lr), 6),
        "n_clusters": new_n_clusters,
        "silhouette": round(float(new_score), 4),
    }
    new_search_log.append(new_log_entry)
    print(f"  → {new_n_clusters} clusters, silhouette={new_score:.4f}")

    if new_score > new_best_score:
        new_best_score = new_score
        new_best_results = {
            'model':                 new_model_stage2,
            'model_stage1':          results['model_stage1'],
            'latents':               new_latents,
            'latents_2d':            new_latents_2d,
            'cluster_labels':        new_cluster_labels,
            'transition_frames':     new_transition_frames,
            'windows':               new_windows,
            'window_segment_labels': new_window_segment_labels,
            'losses':                losses,
            'positions':             positions,
            'lossgraph_stage1':      results['lossgraph_stage1'],
            'lossgraph_stage2':      new_lossgraph_stage2,
            'smoothed_losses':       new_smoothed,
            '_percentile':           percentile,
            '_quantile':             quantile,
            '_lr':                   lr,
        }
        print(f"  ** NEW BEST (silhouette={new_score:.4f}) **")

    del new_model_stage2, new_latents, new_latents_2d, new_cluster_labels, new_ms_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return -new_score

In [ ]:
# ── run optimization ───────────────────────────────────────────────────────
new_bayes_result = gp_minimize(
    func             = new_objective,
    dimensions       = search_space,
    n_calls          = MAX_ITER,
    n_initial_points = min(MAX_ITER, 5),
    random_state     = 42,
    verbose          = False,
)



In [ ]:
# ── summary ────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("OPTIMIZATION COMPLETE")
print(f"{'='*60}")
print(f"\nSearch log:")
for entry in new_search_log:
    marker = " <-- BEST" if entry['silhouette'] == new_best_score else ""
    print(f"  Iter {entry['iteration']:2d}: "
          f"percentile={entry['percentile']:5.2f}, "
          f"quantile={entry['quantile']:.4f}, "
          f"lr={entry['lr']:.6f}  "
          f"→  {entry['n_clusters']} clusters, "
          f"silhouette={entry['silhouette']:.4f}{marker}")

print(f"\nBest parameters:")
print(f"  percentile:  {new_best_results['_percentile']:.2f}")
print(f"  quantile:    {new_best_results['_quantile']:.4f}")
print(f"  lr:          {new_best_results['_lr']:.6f}")
print(f"  n_clusters:  {len(np.unique(new_best_results['cluster_labels']))}")
print(f"  silhouette:  {new_best_score:.4f}")

# ── set results to best ────────────────────────────────────────────────────
new_results = new_best_results

# Save the model
torch.save(new_results['model'].state_dict(), os.path.join('./', "new_model_stage2.pth"))

with open("new_lossgraph_stage2.json", "w") as file:
    json.dump(new_results['lossgraph_stage2'], file)

In [ ]:
with open("new_best_results.pkl", "wb") as file:
    pickle.dump(new_results, file)

In [ ]:
embedded = new_results['latents']
labels = new_results['cluster_labels']
plt.figure(figsize=(8, 6))
plt.scatter(embedded[:, 0], embedded[:, 1])
plt.title("Latent Space")
plt.xlabel("1")
plt.ylabel("2")
plt.show()


"""latents = results['latents']
normed  = normalize(latents, norm='l2')
labels = results['cluster_labels']
# UMAP to 2D
reducer   = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.5, random_state=42)
embedded  = reducer.fit_transform(normed)
"""

embedded = new_results['latents']
labels = new_results['cluster_labels']
plt.figure(figsize=(8, 6))
plt.scatter(embedded[:, 0], embedded[:, 1],
            c=labels, cmap='tab10', s=15, alpha=0.8)
plt.title("Clusters: quantile=0.1078, n_clusters={7}")
plt.colorbar(label='Cluster')
plt.xlabel("1")
plt.ylabel("2")
plt.show()

In [ ]:
# ── plot ───────────────────────────────────────────────────────────────────
new_latents_2d         = new_results['latents_2d']
new_cluster_labels = new_results['cluster_labels']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].scatter(new_latents_2d[:, 0], new_latents_2d[:, 1],
                c=rat_window_ids[:len(new_cluster_labels)],
                cmap='tab10', alpha=0.5, s=10)
axes[0].set_title("Coloured by Rat ID")

scatter = axes[1].scatter(new_latents_2d[:, 0], new_latents_2d[:, 1],
                           c=new_cluster_labels, cmap='tab10', alpha=0.5, s=10)
axes[1].set_title(f"Predicted Clusters (n={len(np.unique(new_cluster_labels))})")
plt.colorbar(scatter, ax=axes[1])

window_x_positions = np.array([
    raw_clean[starts[i]:starts[i] + len(results['windows'][i]), 1, 0].mean()
    for i in range(len(results['windows']))
])
sc = axes[2].scatter(new_latents_2d[:, 0], new_latents_2d[:, 1],
                      c=window_x_positions, cmap='RdYlBu', alpha=0.5, s=10)
axes[2].set_title("Coloured by Arena X Position")
plt.colorbar(sc, ax=axes[2])

plt.suptitle(f"Best — silhouette={new_best_score:.4f}, "
             f"percentile={new_results['_percentile']:.2f}, "
             f"quantile={new_results['_quantile']:.4f}, "
             f"lr={new_results['_lr']:.6f}")
plt.tight_layout()
plt.show()

### Verify performance

In [ ]:
ground_truth_labels = np.load("rat_movement_large_labels.npy", allow_pickle = True)
raw_sequence = np.load("rat_movement_large.npy")
from scipy.stats import mode

In [ ]:
gt_window_labels = []
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment

# ── Parameters (must match pipeline) ──────────────────────────────────────
min_segment_frames = 30
max_segment_frames = 600

# ── Reconstruct window frame ranges ───────────────────────────────────────
n_frames   = len(raw_sequence)
boundaries = np.unique(
    np.concatenate([[0], results['transition_frames'], [n_frames]])
).astype(int)

gt_window_labels = []
for seg_idx in range(len(boundaries) - 1):
    seg_start = boundaries[seg_idx]
    seg_end   = boundaries[seg_idx + 1]
    seg_len   = seg_end - seg_start

    if seg_len < min_segment_frames:
        continue

    for chunk_start in range(0, seg_len, max_segment_frames):
        abs_start = seg_start + chunk_start
        abs_end   = min(abs_start + max_segment_frames, seg_end)
        chunk_len = abs_end - abs_start

        if chunk_len < min_segment_frames:
            continue

        window_frames = ground_truth_labels[abs_start:abs_end]
        values, counts = np.unique(window_frames, return_counts=True)
        majority = values[np.argmax(counts)]
        gt_window_labels.append(majority)

gt_window_labels = np.array(gt_window_labels)
cluster_labels   = results['cluster_labels']

print(f"GT labels:      {len(gt_window_labels)}")
print(f"Cluster labels: {len(cluster_labels)}")
assert len(gt_window_labels) == len(cluster_labels), \
    f"Mismatched: {len(gt_window_labels)} vs {len(cluster_labels)}"

# ── Convert GT strings to numeric for sklearn metrics ─────────────────────
behaviour_names = np.unique(gt_window_labels)   # e.g. ['explore' 'groom' ...]
beh_to_idx      = {b: i for i, b in enumerate(behaviour_names)}
gt_numeric      = np.array([beh_to_idx[b] for b in gt_window_labels])

# ── Metrics ────────────────────────────────────────────────────────────────
ari = adjusted_rand_score(gt_numeric, cluster_labels)
nmi = normalized_mutual_info_score(gt_numeric, cluster_labels)
print(f"\nARI:    {ari:.3f}  (1.0 = perfect, 0 = random)")
print(f"NMI:    {nmi:.3f}  (1.0 = perfect, 0 = random)")

# ── Build confusion matrix manually (GT rows, cluster columns) ────────────
cluster_ids = np.unique(cluster_labels)
cm = np.zeros((len(behaviour_names), len(cluster_ids)), dtype=int)

for gt, pred in zip(gt_window_labels, cluster_labels):
    cm[beh_to_idx[gt], pred] += 1

# ── Hungarian matching: optimal cluster → behaviour assignment ─────────────
row_ind, col_ind = linear_sum_assignment(-cm)
print("\nOptimal cluster → behaviour mapping:")
for r, c in zip(row_ind, col_ind):
    total    = cm[:, c].sum()
    correct  = cm[r, c]
    print(f"  Cluster {c:2d}  →  {behaviour_names[r]:14s}  "
          f"({correct}/{total} = {correct/total:.1%})")

# ── Purity ─────────────────────────────────────────────────────────────────
purity = np.sum(np.max(cm, axis=0)) / np.sum(cm)
print(f"\nPurity: {purity:.3f}")

# ── Pure windows ───────────────────────────────────────────────────────────
pure_windows = []
for seg_idx in range(len(boundaries) - 1):
    seg_start = boundaries[seg_idx]
    seg_end   = boundaries[seg_idx + 1]
    seg_len   = seg_end - seg_start

    if seg_len < min_segment_frames:
        continue

    for chunk_start in range(0, seg_len, max_segment_frames):
        abs_start = seg_start + chunk_start
        abs_end   = min(abs_start + max_segment_frames, seg_end)
        if (abs_end - abs_start) < min_segment_frames:
            continue
        n_unique = len(np.unique(ground_truth_labels[abs_start:abs_end]))
        pure_windows.append(n_unique == 1)

print(f"Pure windows:   {np.mean(pure_windows):.1%}")

# ── Plot confusion matrix ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
            xticklabels=[f'Cluster {i}' for i in cluster_ids],
            yticklabels=behaviour_names)
ax.set_xlabel("Predicted Cluster")
ax.set_ylabel("Ground Truth Behaviour")
ax.set_title(f"Confusion Matrix  (ARI={ari:.3f},  NMI={nmi:.3f},  Purity={purity:.3f})")
plt.tight_layout()
plt.show()

In [ ]:
# ── Check 1: how pure are your windows? ───────────────────
# if this is low, transition detection is the bottleneck
print(f"Pure windows: {np.mean(pure_windows):.1%}")
# if < 70%, fix transition detection before anything else

# ── Check 2: does the latent space have structure? ─────────
# plot GT labels on the latent space
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
scatter = axes[0].scatter(results['latents'][:, 0], results['latents'][:, 1],
                           c=gt_numeric, cmap='tab10', alpha=0.7, s=20)
axes[0].set_title("Ground Truth Behaviours")
plt.colorbar(scatter, ax=axes[0],
             ticks=range(len(behaviour_names))).set_ticklabels(behaviour_names)

scatter2 = axes[1].scatter(results['latents'][:, 0], results['latents'][:, 1],
                            c=results['cluster_labels'], cmap='tab10', alpha=0.7, s=20)
axes[1].set_title("Predicted Clusters")
plt.colorbar(scatter2, ax=axes[1])
plt.tight_layout()
plt.show()
# if GT colours are jumbled → model isn't learning, fix preprocessing/training
# if GT colours show structure but clusters don't align → fix clustering

# ── Check 3: upper bound ARI if windows were perfect ───────
# assign each window its majority GT label as the prediction
# this tells you the maximum ARI your transition detector allows
from sklearn.metrics import adjusted_rand_score
upper_bound_ari = adjusted_rand_score(gt_numeric, gt_numeric)
print(f"Upper bound ARI (perfect clustering): {upper_bound_ari:.3f}")  # should be 1.0

# more useful — what ARI would you get if you clustered perfectly
# on only the pure windows?
pure_mask = np.array(pure_windows)
if pure_mask.sum() > 0:
    ari_pure = adjusted_rand_score(gt_numeric[pure_mask],
                                   results['cluster_labels'][pure_mask])
    print(f"ARI on pure windows only: {ari_pure:.3f}")

In [ ]:
# ============================================================
# SAVE ALL RESULTS AND FIGURES TO ./results
# ============================================================


RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── 0. Save hyperparameter search log ─────────────────────────────────────
if 'search_log' in dir():
    with open(os.path.join(RESULTS_DIR, "hyperparameter_search.json"), "w") as f:
        json.dump({
            "objective": "maximize ARI",
            "max_iterations": 10,
            "percentile_range": [50, 100],
            "quantile_range": [0.05, 0.5],
            "search_log": search_log,
            "best_percentile": results.get('_percentile', None),
            "best_quantile": results.get('_quantile', None),
        }, f, indent=2)
    print("Saved hyperparameter search log.")

# ── 1. Save model weights ─────────────────────────────────────────────────
torch.save(results['model'].state_dict(), os.path.join(RESULTS_DIR, "model_stage1.pth"))
torch.save(results['model'].state_dict(), os.path.join(RESULTS_DIR, "model_stage2.pth"))
print("Saved model weights.")

# ── 2. Save numerical results ─────────────────────────────────────────────
np.save(os.path.join(RESULTS_DIR, "latents.npy"), results['latents'])
np.save(os.path.join(RESULTS_DIR, "latents_2d.npy"), results['latents_2d'])
np.save(os.path.join(RESULTS_DIR, "cluster_labels.npy"), results['cluster_labels'])
np.save(os.path.join(RESULTS_DIR, "transition_frames.npy"), results['transition_frames'])
np.save(os.path.join(RESULTS_DIR, "losses.npy"), results['losses'])
np.save(os.path.join(RESULTS_DIR, "positions.npy"), results['positions'])
np.save(os.path.join(RESULTS_DIR, "smoothed_losses.npy"), results['smoothed_losses'])
#np.save(os.path.join(RESULTS_DIR, "lossgraph_stage1.npy"), np.array(results['lossgraph_stage1']))
#np.save(os.path.join(RESULTS_DIR, "lossgraph_stage2.npy"), np.array(results['lossgraph_stage2']))
print("Saved numerical results.")

# ── 3. Save window metadata ───────────────────────────────────────────────
window_lengths = [len(w) for w in results['windows']]
window_meta = {
    "n_windows": len(results['windows']),
    "min_length": int(min(window_lengths)),
    "max_length": int(max(window_lengths)),
    "mean_length": float(np.mean(window_lengths)),
    "n_transitions": int(len(results['transition_frames'])),
    "n_clusters": int(len(np.unique(results['cluster_labels']))),
    "best_percentile": results.get('_percentile', None),
    "best_quantile": results.get('_quantile', None),
}
with open(os.path.join(RESULTS_DIR, "window_meta.json"), "w") as f:
    json.dump(window_meta, f, indent=2)
print("Saved window metadata.")

# ── 4. Save clustering metrics ────────────────────────────────────────────
behaviour_names = np.unique(gt_window_labels)
beh_to_idx = {b: i for i, b in enumerate(behaviour_names)}
gt_numeric = np.array([beh_to_idx[b] for b in gt_window_labels])
cluster_ids = np.unique(results['cluster_labels'])
cm = np.zeros((len(behaviour_names), len(cluster_ids)), dtype=int)
for gt, pred in zip(gt_window_labels, results['cluster_labels']):
    cm[beh_to_idx[gt], pred] += 1

ari = adjusted_rand_score(gt_numeric, results['cluster_labels'])
nmi = normalized_mutual_info_score(gt_numeric, results['cluster_labels'])
purity = np.sum(np.max(cm, axis=0)) / np.sum(cm)

row_ind, col_ind = linear_sum_assignment(-cm)
mapping = {}
for r, c in zip(row_ind, col_ind):
    total = int(cm[:, c].sum())
    correct = int(cm[r, c])
    mapping[f"Cluster {c}"] = {
        "behaviour": str(behaviour_names[r]),
        "correct": correct,
        "total": total,
        "accuracy": correct / total if total > 0 else 0.0,
    }

metrics = {
    "ARI": round(float(ari), 4),
    "NMI": round(float(nmi), 4),
    "Purity": round(float(purity), 4),
    "cluster_behaviour_mapping": mapping,
    "confusion_matrix": cm.tolist(),
    "behaviour_names": behaviour_names.tolist(),
    "cluster_ids": cluster_ids.tolist(),
}
with open(os.path.join(RESULTS_DIR, "clustering_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved clustering metrics.")

# ── 5. Save all figures ───────────────────────────────────────────────────

# 5a. Training loss curves
#fig, axes = plt.subplots(1, 2, figsize=(14, 5))
#axes[0].plot(results['lossgraph_stage1'], 'b-', linewidth=1.5)
#axes[0].set_title("Stage 1: Fixed Windows Training Loss")
#axes[0].set_xlabel("Epoch")
#axes[0].set_ylabel("MSE Loss")
#axes[0].grid(True, alpha=0.3)

#axes[1].plot(results['lossgraph_stage2'], 'r-', linewidth=1.5)
#axes[1].set_title("Stage 2: Variable Windows Training Loss")
#axes[1].set_xlabel("Epoch")
#axes[1].set_ylabel("MSE Loss")
#axes[1].grid(True, alpha=0.3)

#plt.tight_layout()
#fig.savefig(os.path.join(RESULTS_DIR, "training_loss.png"), dpi=150, bbox_inches="tight")
#plt.close(fig)
#print("Saved training_loss.png")

# 5b. Reconstruction loss signal and transitions
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(results['positions'], results['losses'], 'b.', markersize=2, alpha=0.3, label="Raw loss")
ax.plot(results['positions'], results['smoothed_losses'], 'r-', linewidth=1.5, label="Smoothed loss")
for tf in results['transition_frames']:
    ax.axvline(x=tf, color='g', alpha=0.15, linewidth=0.5)
ax.set_title("Reconstruction Loss & Detected Transitions")
ax.set_xlabel("Frame")
ax.set_ylabel("MSE Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "reconstruction_loss.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved reconstruction_loss.png")

# 5c. Latent space scatter (no labels)
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(results['latents'][:, 0], results['latents'][:, 1], s=10, alpha=0.6)
ax.set_title("Latent Space")
ax.set_xlabel("Dim 1")
ax.set_ylabel("Dim 2")
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "latent_space.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved latent_space.png")

# 5d. Clustered latent space
n_clusters = len(np.unique(results['cluster_labels']))
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(results['latents'][:, 0], results['latents'][:, 1],
                     c=results['cluster_labels'], cmap='tab10', s=15, alpha=0.8)
ax.set_title(f"Clusters (n={n_clusters})\n"
             f"percentile={results.get('_percentile', 'N/A'):.1f}, quantile={results.get('_quantile', 'N/A'):.3f}")
plt.colorbar(scatter, ax=ax, label='Cluster')
ax.set_xlabel("Dim 1")
ax.set_ylabel("Dim 2")
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "clustered_latent_space.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved clustered_latent_space.png")

# 5e. Confusion matrix
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
            xticklabels=[f'Cluster {i}' for i in cluster_ids],
            yticklabels=behaviour_names)
ax.set_xlabel("Predicted Cluster")
ax.set_ylabel("Ground Truth Behaviour")
ax.set_title(f"Confusion Matrix (ARI={ari:.3f}, NMI={nmi:.3f}, Purity={purity:.3f})\n"
             f"percentile={results.get('_percentile', 'N/A'):.1f}, quantile={results.get('_quantile', 'N/A'):.3f}")
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved confusion_matrix.png")

# 5f. Diagnostic: GT vs Predicted on latent space
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
scatter = axes[0].scatter(results['latents'][:, 0], results['latents'][:, 1],
                          c=gt_numeric, cmap='tab10', alpha=0.7, s=20)
axes[0].set_title("Ground Truth Behaviours")
plt.colorbar(scatter, ax=axes[0],
             ticks=range(len(behaviour_names))).set_ticklabels(behaviour_names)

scatter2 = axes[1].scatter(results['latents'][:, 0], results['latents'][:, 1],
                           c=results['cluster_labels'], cmap='tab10', alpha=0.7, s=20)
axes[1].set_title("Predicted Clusters")
plt.colorbar(scatter2, ax=axes[1])
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "diagnostic_gt_vs_predicted.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved diagnostic_gt_vs_predicted.png")

# 5g. Pure windows diagnostic bar chart
pure_mask = np.array(pure_windows)
ari_pure = adjusted_rand_score(gt_numeric[pure_mask], results['cluster_labels'][pure_mask]) if pure_mask.sum() > 0 else 0
fig, ax = plt.subplots(figsize=(8, 5))
categories = ["All Windows", "Pure Windows Only"]
ari_values = [ari, ari_pure]
nmi_values = [nmi, nmi]
x = np.arange(len(categories))
width = 0.35
bars1 = ax.bar(x - width/2, ari_values, width, label='ARI', color='steelblue')
bars2 = ax.bar(x + width/2, nmi_values, width, label='NMI', color='coral')
ax.set_ylabel("Score")
ax.set_title("Clustering Performance: All vs Pure Windows")
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
ax.set_ylim(0, 1.1)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "performance_comparison.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved performance_comparison.png")

print(f"\nAll results saved to ./{RESULTS_DIR}/")